In [ ]:
# Load environment variables

import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# Import required LlamaIndex and Chroma components
from llama_index.core import (
    SimpleDirectoryReader,
    VectorStoreIndex,
    Settings,
    StorageContext,
    PromptTemplate,
)
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.retrievers import QueryFusionRetriever
from llama_index.retrievers.bm25 import BM25Retriever
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.gemini import Gemini
from llama_index.embeddings.ollama import OllamaEmbedding
from llama_index.core.postprocessor import LLMRerank
from llama_index.core import set_global_handler
from llama_index.core.ingestion import IngestionPipeline, IngestionCache
import chromadb

resource module not available on Windows


d:\GEN AI PROJECTS\RAG4Diabetes\rag4diabetes-env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Read Gemini API key
api_key = os.getenv("GEMINI_API_KEY")

In [4]:
# Configure LLM and Embedding globally
embed_model = OllamaEmbedding(
    model_name="nomic-embed-text:latest",
    base_url="http://localhost:11434",
)

llm = Gemini(
    model="models/gemini-2.5-flash",
    api_key=api_key
)

Settings.embed_model = embed_model
Settings.llm = llm

D:\Data Science\Temp\ipykernel_1964\3983992690.py:7: DeprecationWarning: Call to deprecated class Gemini. (Should use `llama-index-llms-google-genai` instead, using Google's latest unified SDK. See: https://docs.llamaindex.ai/en/stable/examples/llm/google_genai/)
  llm = Gemini(


In [5]:
# Load documents from data directory
documents = SimpleDirectoryReader(r"D:\GEN AI PROJECTS\RAG4Diabetes\data", recursive=True).load_data()

In [6]:
# Split documents into sentence-based chunks
splitter = SentenceSplitter(
    chunk_size=256,
    chunk_overlap=50
)
nodes = splitter.get_nodes_from_documents(documents)

In [7]:
# Initialize persistent Chroma database
chroma_client = chromadb.PersistentClient(path="./chroma_db")
chroma_collection = chroma_client.get_or_create_collection("diabetes_vectors")

2025-12-16 22:27:01,675 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.


In [8]:
# Create vector store and storage context
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [9]:
# Build vector index using nodes and storage context
index = VectorStoreIndex(
    nodes=nodes,
    storage_context=storage_context
)

2025-12-16 22:27:04,604 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:27:05,175 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:27:06,190 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:27:07,343 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:27:08,504 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:27:09,101 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:27:09,276 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:27:09,488 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:27:09,630 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:27:09,778 - INFO - HTTP Request: POST http://localhost:1143

In [10]:
# Prompt for answer synthesis
qa_prompt = PromptTemplate(
    "Context:\n{context_str}\n\n"
    "Instructions:\n"
    "- Answer using ONLY the information from the context.\n"
    "- Do NOT use external knowledge.\n"
    "- If information is missing, say exactly:\n"
    "  \"I do not have the information based on the provided context.\"\n"
    "- Answer in concise bullet points.\n\n"
    "Query: {query_str}\n"
    "Answer:"
)

In [11]:
# Create vector and BM25 retrievers
vector_retriever = index.as_retriever(similarity_top_k=5)

bm25_retriever = BM25Retriever.from_defaults(
    nodes=nodes,
    similarity_top_k=5
)

2025-12-16 22:32:03,575 - DEBUG - Building index from IDs objects


In [12]:
# Fuse retrievers using query expansion
fusion_retriever = QueryFusionRetriever(
    retrievers=[vector_retriever, bm25_retriever],
    similarity_top_k=5,
    num_queries=4,
    use_async=True,
    verbose=True
)

In [13]:
# Initialize reranker (same number of nodes -> only reorder)
reranker = LLMRerank(
    top_n=5   # same as similarity_top_k -> re-rank only
)

In [19]:
import nest_asyncio
nest_asyncio.apply()

In [20]:
# Create query engine with reranker attached
query_engine = RetrieverQueryEngine.from_args(
    retriever=fusion_retriever,
    node_postprocessors=[reranker]
)

In [21]:
# Inject custom prompt into response synthesizer
query_engine.update_prompts({
    "response_synthesizer:text_qa_template": qa_prompt
})

In [22]:
# Run query against the RAG pipeline
response = query_engine.query(
    "According to standard medical classification, explain ONLY the main primary types of diabetes mellitus. Do not include subtypes, causes, historical terms, or descriptive forms."
)


Generated queries:
standard medical classification main types diabetes mellitus
primary types diabetes mellitus official classification
WHO ADA classification major types diabetes mellitus


2025-12-16 22:51:57,613 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:51:58,077 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:51:58,082 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2025-12-16 22:51:58,086 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


In [23]:
# Print final response
print(response)

According to the revised classification:
*   Type 1 diabetes
*   Type 2 diabetes
*   Hybrid types of diabetes
*   Unclassified diabetes
